In [130]:
import torch 
import numpy as np 
import pickle
import tqdm 

In [131]:
model = torch.load("proto.pt", weights_only = False)

In [132]:
def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict
def sample_data(num_classes: int, shot_count: int):
    #hard coded classes for CIFAR10 -> 0-9
    data_ret = {}
    classes = np.random.choice(unique_labels, num_classes, replace = False)
    used_indices = []
    for i in classes:
        data_indexes = np.random.choice(classes_to_index[i], shot_count, replace = False)
        used_indices.append(data_indexes)
        image_dat = data[data_indexes]
        data_ret[i] = image_dat
    used_indices = np.concatenate(used_indices)
    return data_ret, used_indices


In [133]:
unique_labels = [0,1,2,3,4,5,6,7,8,9]
raw_data = unpickle(r"C:\Users\simon\Coding\ML\Prototypical NN\cifar-10-batches-py\test_batch")
data = raw_data[b"data"]
all_labels = raw_data[b"labels"]
data = data.reshape(-1, 3, 32, 32)
data = torch.tensor(data, dtype=torch.float32) / 255.0
labels = torch.tensor(all_labels)
classes_to_index = {}
for i in unique_labels:
    classes_to_index[i] = np.where(labels == i)[0]

In [134]:
num_classes = 5
shot = 5

In [135]:
acc_count = 0
model.eval()
with torch.no_grad():
    for i in tqdm.tqdm(range(2000)):
        samp = sample_data(num_classes,shot)
        support_set = samp[0]
        used_indices = samp[1]
        classes = list(support_set.keys())
        prototypes = {}
        valid_query_indices = []
        for j in classes:
            #prototype calculation
            prototype = torch.sum(model.forward(support_set[j]), dim = 0)
            prototype /= shot
            prototypes[j] = prototype
    
            #query set generation: calculate the rest of valid images to ensure support and queries are disjoint
            valid_query_indices.append(np.setdiff1d(classes_to_index[j], used_indices))
        valid_query_indices = np.concatenate(valid_query_indices)
        query_idx = np.random.choice(valid_query_indices, 1)[0]
        query_img = model.forward(data[query_idx].unsqueeze(0))[0]
        min_dist = torch.dist(list(prototypes.values())[0], query_img)
        guess_class = list(prototypes.keys())[0]
        for j in prototypes.items():
            distance = torch.dist(j[1], query_img)
            if (distance < min_dist):
                min_dist = distance
                guess_class = j[0]
        if (labels[query_idx] == guess_class):
            acc_count += 1
        
print(acc_count/2000)

    
    


100%|██████████| 2000/2000 [00:34<00:00, 58.58it/s]

0.6195
